# Phase 1: Data Cleaning and Preprocessing

### Recipe Dataset Preprocessing Pipeline

This document describes the preprocessing steps applied to the
`RAW_recipes.csv` dataset to make it suitable for LLM-based retrieval
and downstream reasoning tasks (LangChain / RAG).

---

#### 1. Data Loading

The raw dataset is loaded from disk using pandas.

- Source file: `Data/RAW_recipes.csv`
- Format: CSV

```python
raw_data = pd.read_csv("Data/RAW_recipes.csv")
```
#### 2. Column Pruning

Columns not required for semantic retrieval or reasoning are removed.

Dropped columns:

- `id`

- `contributor_id`

- `submitted`

This reduces noise and dataset size while preserving all useful content.

#### 3. Parsing Stringified List Columns

Several columns are stored as string representations of Python lists
(e.g. "['salt', 'butter']").

The following columns are converted into actual Python lists using `ast.literal_eval`:

- `tags`

- `steps`

- `ingredients`

- `nutrition`

This ensures correct typing and prevents downstream parsing errors.

#### 4. Nutrition Normalization

The nutrition column (a list of numeric values) is expanded into explicit scalar columns based on the dataset schema:

- calories

- total_fat_pdv

- sugar_pdv

- sodium_pdv

- protein_pdv

- saturated_fat_pdv

- carbs_pdv

After expansion, the original `nutrition` column is dropped.

This converts nutrition data into fixed-width, numeric features suitable for filtering and aggregation.

#### 5. Light Text Hygiene

Minimal, human-safe cleaning is applied to textual fields:

- Removes newlines, carriage returns, and tabs

- Trims extra whitespace

- Preserves punctuation, casing, and semantics

Applied to:

- `name`

- `description`

This improves readability without harming semantic meaning.

#### 6. Canonical Document Construction

A single, human-readable text field (document) is constructed for each recipe, combining:

- Recipe name

- Description

- Ingredients list

- Step-by-step instructions

This column is intended for:

- Embedding generation

- Semantic retrieval

- RAG-based question answering

The final dataset contains both:

- A unified text representation (document)

- Structured numeric metadata (nutrition, time, counts)

#### Result

The processed dataset is:

- Schema-stable

- LLM-friendly

- Ready for LangChain document ingestion

Suitable for filtering, retrieval, and reasoning

In [2]:
import pandas as pd
import re
import ast


In [6]:
# --------------------------------------------------
# 1. Load raw data
# --------------------------------------------------
raw_data = pd.read_csv("./raw/RAW_recipes.csv")

# --------------------------------------------------
# 2. Drop truly unnecessary columns
# --------------------------------------------------
raw_data = raw_data.drop(
    columns=["id", "contributor_id", "submitted"]
)

In [7]:
# --------------------------------------------------
# 3. Parse stringified list columns
# --------------------------------------------------
list_cols = ["tags", "steps", "ingredients", "nutrition"]

for col in list_cols:
    raw_data[col] = raw_data[col].apply(ast.literal_eval)


In [8]:
# -------------------------------------------------------------------------------------------
# 4. Expand nutrition into scalar columns (this information in given in the dataset datacard
# -------------------------------------------------------------------------------------------
nutrition_cols = [
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbs_pdv"
]

raw_data[nutrition_cols] = pd.DataFrame(
    raw_data["nutrition"].tolist(),
    index=raw_data.index
)
# dropping the original nutrition col as its no longer useful now
raw_data = raw_data.drop(columns=["nutrition"])

In [9]:
# --------------------------------------------------
# 5. Light text hygiene (human-safe)
# --------------------------------------------------

def clean_text(s):
    return (
        re.sub(r"\s+", " ",
            str(s)
            .replace("\r", " ")
            .replace("\n", " ")
            .replace("\t", " ")
        )
        .strip()
    )


raw_data["name"] = raw_data["name"].apply(clean_text)
raw_data["description"] = raw_data["description"].apply(clean_text)


In [10]:
# --------------------------------------------------
# 6. Build ONE canonical document column
# --------------------------------------------------
raw_data["document"] = (
    "Recipe: " + raw_data["name"] + "\n\n"

    "Description: " + raw_data["description"] + "\n\n"
    "Ingredients:\n" +
    raw_data["ingredients"].apply(lambda x: ", ".join(x)) + "\n\n"
    "Steps:\n" +
    raw_data["steps"].apply(lambda x: " ".join(f"{i+1}. {s}" for i, s in enumerate(x)))
)

In [11]:
raw_data.to_csv("./processed/PROCESSED_recipes.csv", index=False)
raw_data.head(5)

,name,minutes,tags,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv,carbs_pdv,document
0,arriba baked winter squash mexican style,55,"[60-minutes-or-less, time-to-make, course, mai...",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0,Recipe: arriba baked winter squash mexican sty...
1,a bit different breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0,Recipe: a bit different breakfast pizza\n\nDes...
2,all in the kitchen chili,130,"[time-to-make, course, preparation, main-dish,...",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0,Recipe: all in the kitchen chili\n\nDescriptio...
3,alouette potatoes,45,"[60-minutes-or-less, time-to-make, course, mai...",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,368.1,17.0,10.0,2.0,14.0,8.0,20.0,Recipe: alouette potatoes\n\nDescription: this...
4,amish tomato ketchup for canning,190,"[weeknight, time-to-make, course, main-ingredi...",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,352.9,1.0,337.0,23.0,3.0,0.0,28.0,Recipe: amish tomato ketchup for canning\n\nDe...
